In [5]:
import ccxt
import pandas as pd
import numpy as np
import ta
import time

from tabulate import tabulate

In [6]:
exchange = ccxt.binance({
    "enableRateLimit": True,
    "options": {
        "defaultType": "future"
    }
})

In [7]:
markets = exchange.load_markets()

usdt_pairs = [

    symbol

    for symbol, market in markets.items()

    if market['quote'] == 'USDT'
    and market['contract']
    and market['swap']
    and market['active']
]

print(f"Total active USDT futures pairs: {len(usdt_pairs)}")

Total active USDT futures pairs: 579


In [8]:
filtered_pairs = []

for symbol in usdt_pairs:

    try:

        ticker = exchange.fetch_ticker(symbol)

        if (
            ticker['quoteVolume']
            and ticker['quoteVolume'] > 5_000_000
        ):

            filtered_pairs.append(symbol)

            print(f"Added: {symbol}")

    except:
        pass

print(f"\nHigh-liquidity pairs: {len(filtered_pairs)}")

Added: BTC/USDT:USDT
Added: ETH/USDT:USDT
Added: BCH/USDT:USDT
Added: XRP/USDT:USDT
Added: LTC/USDT:USDT
Added: TRX/USDT:USDT
Added: ETC/USDT:USDT
Added: LINK/USDT:USDT
Added: XLM/USDT:USDT
Added: ADA/USDT:USDT
Added: XMR/USDT:USDT
Added: DASH/USDT:USDT
Added: ZEC/USDT:USDT
Added: BNB/USDT:USDT
Added: ATOM/USDT:USDT
Added: ONT/USDT:USDT
Added: VET/USDT:USDT
Added: THETA/USDT:USDT
Added: ALGO/USDT:USDT
Added: COMP/USDT:USDT
Added: DOGE/USDT:USDT
Added: DOT/USDT:USDT
Added: CRV/USDT:USDT
Added: TRB/USDT:USDT
Added: RUNE/USDT:USDT
Added: SOL/USDT:USDT
Added: STORJ/USDT:USDT
Added: UNI/USDT:USDT
Added: AVAX/USDT:USDT
Added: ENJ/USDT:USDT
Added: NEAR/USDT:USDT
Added: AAVE/USDT:USDT
Added: FIL/USDT:USDT
Added: AXS/USDT:USDT
Added: ZEN/USDT:USDT
Added: GRT/USDT:USDT
Added: CHZ/USDT:USDT
Added: SAND/USDT:USDT
Added: HBAR/USDT:USDT
Added: MTL/USDT:USDT
Added: 1000SHIB/USDT:USDT
Added: GTC/USDT:USDT
Added: DYDX/USDT:USDT
Added: GALA/USDT:USDT
Added: AR/USDT:USDT
Added: LPT/USDT:USDT
Added: ENS/U

In [9]:
def get_ohlcv(symbol, timeframe='15m', limit=500):

    ohlcv = exchange.fetch_ohlcv(
        symbol,
        timeframe=timeframe,
        limit=limit
    )

    df = pd.DataFrame(
        ohlcv,
        columns=[
            'timestamp',
            'open',
            'high',
            'low',
            'close',
            'volume'
        ]
    )

    df['timestamp'] = pd.to_datetime(
        df['timestamp'],
        unit='ms'
    )

    return df

In [10]:
def add_rsi(df, length=14):

    delta = df['close'].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    alpha = 1 / length

    avg_gain = gain.ewm(
        alpha=alpha,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=alpha,
        adjust=False
    ).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    df['rsi'] = rsi

    return df

In [11]:
def add_stochastic(df, periodK=14, smoothK=3, smoothD=3):

    low_min = df['low'].rolling(periodK).min()

    high_max = df['high'].rolling(periodK).max()

    stoch = (
        100
        * (df['close'] - low_min)
        / (high_max - low_min)
    )

    k = stoch.rolling(smoothK).mean()

    d = k.rolling(smoothD).mean()

    df['stoch_k'] = k

    df['stoch_d'] = d

    return df

In [12]:
def add_stoch_rsi(
    df,
    lengthRSI=14,
    lengthStoch=14,
    smoothK=3,
    smoothD=3
):

    delta = df['close'].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    alpha = 1 / lengthRSI

    avg_gain = gain.ewm(
        alpha=alpha,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=alpha,
        adjust=False
    ).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    rsi_min = rsi.rolling(lengthStoch).min()

    rsi_max = rsi.rolling(lengthStoch).max()

    stoch = (
        100
        * (rsi - rsi_min)
        / (rsi_max - rsi_min)
    )

    k = stoch.rolling(smoothK).mean()

    d = k.rolling(smoothD).mean()

    df['stoch_rsi_k'] = k

    df['stoch_rsi_d'] = d

    return df

In [13]:
def add_trend(df):

    df['ma99'] = ta.trend.ema_indicator(
        close=df['close'],
        window=99
    )

    return df

In [14]:
def check_signal(df):

    last = df.iloc[-1]

    # ============================================
    # INDICATORS
    # ============================================

    rsi = last['rsi']

    stoch_rsi = last['stoch_rsi_k']

    stoch_k = last['stoch_d']

    close = last['close']

    ma99 = last['ma99']

    # ============================================
    # TREND
    # ============================================

    bullish_trend = close > ma99

    bearish_trend = close < ma99

    # ============================================
    # LONG SETUP
    # ============================================

    pullback_long = (

        rsi < 45

        and stoch_rsi < 45

        and stoch_k < 45
    )

    momentum_up = (
        stoch_k > stoch_rsi
    )

    long_signal = (

        bullish_trend

        and pullback_long

        and momentum_up
    )

    # ============================================
    # SHORT SETUP
    # ============================================

    pullback_short = (

        rsi > 55

        and stoch_rsi > 55

        and stoch_k > 55
    )

    momentum_down = (
        stoch_k < stoch_rsi
    )

    short_signal = (

        bearish_trend

        and pullback_short

        and momentum_down
    )

    # ============================================
    # FINAL SIGNALS
    # ============================================

    if long_signal:
        return "LONG"

    elif short_signal:
        return "SHORT"

    return None

In [15]:
timeframes = ['15m', '1h', '4h']

def analyze_symbol(symbol):

    long_agree = []

    short_agree = []

    for tf in timeframes:

        try:

            df = get_ohlcv(symbol, tf)

            # REMOVE CURRENT OPEN CANDLE
            df = df.iloc[:-1]

            # ===============================
            # ADD INDICATORS
            # ===============================

            df = add_rsi(df)

            df = add_stochastic(df)

            df = add_stoch_rsi(df)

            df = add_trend(df)

            # ===============================
            # SIGNAL
            # ===============================

            signal = check_signal(df)

            if signal == "LONG":
                long_agree.append(tf)

            elif signal == "SHORT":
                short_agree.append(tf)

        except Exception as e:

            print(f"Error {symbol} {tf}: {e}")

    return long_agree, short_agree

In [16]:
def scan_market(pairs):

    results = []

    for i, symbol in enumerate(pairs, 1):

        print(f"[{i}/{len(pairs)}] Scanning {symbol}")

        try:

            long_agree, short_agree = analyze_symbol(symbol)

            # ==================================
            # LONG RESULTS
            # ==================================

            if len(long_agree) >= 2:

                results.append({

                    'Symbol': symbol,

                    'Direction': 'LONG',

                    'Agree TFs': ', '.join(long_agree),

                    'Strength': len(long_agree)
                })

            # ==================================
            # SHORT RESULTS
            # ==================================

            if len(short_agree) >= 2:

                results.append({

                    'Symbol': symbol,

                    'Direction': 'SHORT',

                    'Agree TFs': ', '.join(short_agree),

                    'Strength': len(short_agree)
                })

        except Exception as e:

            print(f"Scan Error {symbol}: {e}")

    return results

In [17]:
signals = scan_market(filtered_pairs)

[1/273] Scanning BTC/USDT:USDT
[2/273] Scanning ETH/USDT:USDT
[3/273] Scanning BCH/USDT:USDT
[4/273] Scanning XRP/USDT:USDT
[5/273] Scanning LTC/USDT:USDT
[6/273] Scanning TRX/USDT:USDT
[7/273] Scanning ETC/USDT:USDT
[8/273] Scanning LINK/USDT:USDT
[9/273] Scanning XLM/USDT:USDT
[10/273] Scanning ADA/USDT:USDT
[11/273] Scanning XMR/USDT:USDT
[12/273] Scanning DASH/USDT:USDT
[13/273] Scanning ZEC/USDT:USDT
[14/273] Scanning BNB/USDT:USDT
[15/273] Scanning ATOM/USDT:USDT
[16/273] Scanning ONT/USDT:USDT
[17/273] Scanning VET/USDT:USDT
[18/273] Scanning THETA/USDT:USDT
[19/273] Scanning ALGO/USDT:USDT
[20/273] Scanning COMP/USDT:USDT
[21/273] Scanning DOGE/USDT:USDT
[22/273] Scanning DOT/USDT:USDT
[23/273] Scanning CRV/USDT:USDT
[24/273] Scanning TRB/USDT:USDT
[25/273] Scanning RUNE/USDT:USDT
[26/273] Scanning SOL/USDT:USDT
[27/273] Scanning STORJ/USDT:USDT
[28/273] Scanning UNI/USDT:USDT
[29/273] Scanning AVAX/USDT:USDT
[30/273] Scanning ENJ/USDT:USDT
[31/273] Scanning NEAR/USDT:USDT
[32/

In [18]:
results_df = pd.DataFrame(signals)

if len(results_df) > 0:

    results_df = results_df.sort_values(
        by='Strength',
        ascending=False
    )

    print(
        tabulate(
            results_df,
            headers='keys',
            tablefmt='fancy_grid',
            showindex=False
        )
    )

else:

    print("No setups found.")

No setups found.
